# 20 — Sandbox, MCP & Agentic Supply-Chain Security

## Learning requirements
- sandbox protects host boundaries but does not make untrusted instructions safe;
- filesystem, process, network and secret boundaries phải thiết kế riêng;
- network egress là data-exfiltration boundary;
- MCP server/tool là third-party capability boundary, không phải trusted automatically;
- package/model/prompt/skill/MCP/tool dependencies đều thuộc agentic supply chain;
- production shell/code execution cần isolation + quotas + audit + cleanup.

## Production rule
Không chạy agent shell/code execution trực tiếp trên application host. Local filesystem/shell backends chỉ phù hợp môi trường được kiểm soát; production cần sandbox/container/VM isolation phù hợp risk model.

## Sandbox security model

```text
Agent Runtime
    |
    | bounded execute request
    v
+--------------------------+
| Sandbox                  |
|  filesystem root         |
|  CPU / memory / timeout  |
|  no host secrets         |
|  restricted network      |
+--------------------------+
          |
       allow-list
          |
          v
 approved external services
```

Review:
- fresh-per-thread vs persistent sandbox;
- writable paths;
- symlink/path traversal;
- process limits;
- output-size limits;
- network/DNS egress;
- secret injection;
- TTL and cleanup.

In [ ]:
from dataclasses import dataclass
from urllib.parse import urlparse

@dataclass(frozen=True)
class SandboxPolicy:
    allowed_hosts: frozenset[str]
    max_seconds: int
    max_output_bytes: int
    allow_shell: bool

POLICY = SandboxPolicy(
    allowed_hosts=frozenset({"api.github.com"}),
    max_seconds=30,
    max_output_bytes=100_000,
    allow_shell=False,
)

def allow_outbound_url(url: str, policy: SandboxPolicy = POLICY) -> bool:
    parsed = urlparse(url)
    return parsed.scheme == "https" and parsed.hostname in policy.allowed_hosts

assert allow_outbound_url("https://api.github.com/repos/example/demo")
assert not allow_outbound_url("http://api.github.com/repos/example/demo")
assert not allow_outbound_url("https://untrusted.example/collect")

## MCP trust boundary

MCP integration cần kiểm soát:
- server identity/authentication;
- approved server registry;
- tool allow-list per agent/use case;
- input schema validation;
- user/tenant/session isolation;
- least-privilege credentials server-side;
- tool-result provenance/untrusted labeling;
- timeout/retry/rate limits;
- destructive tool HITL;
- server/version change review.

Không để agent tự discover arbitrary Internet MCP server rồi tự cấp quyền sử dụng.

In [ ]:
APPROVED_MCP = {
    "github-readonly": {
        "allowed_tools": {"get_file", "search_code", "list_commits"},
        "write": False,
    },
    "project-store": {
        "allowed_tools": {"get_project", "search_documents"},
        "write": False,
    },
}

def approve_mcp_tool(server: str, tool: str) -> bool:
    config = APPROVED_MCP.get(server)
    return bool(config and tool in config["allowed_tools"])

assert approve_mcp_tool("github-readonly", "search_code")
assert not approve_mcp_tool("github-readonly", "delete_repo")
assert not approve_mcp_tool("unknown-server", "search_code")

## Agentic supply-chain inventory

Inventory và review:
- Python/JS dependencies + lockfiles;
- LangChain/LangGraph/Deep Agents versions;
- model provider/model ID changes;
- prompts/system policies;
- skills and skill scripts;
- MCP servers;
- tool descriptions/schemas;
- container base images;
- vector-store ingestion sources.

Một tool description, skill hoặc MCP server update có thể thay agent behavior dù application code không đổi.

## Required output
- `artifacts/security/sandbox-design.md`
- `artifacts/security/mcp-trust-policy.md`
- `artifacts/security/supply-chain-inventory.md`
- tests cho path/network/tool allow-lists.

## Done criteria
- Host secrets không accessible từ execution sandbox.
- Default network egress policy được define.
- Unknown MCP server/tool bị deny.
- Dependency/tool/prompt changes có review + regression-security-eval path.